In [0]:
# import dlt
# from pyspark.sql.functions import *
# from pyspark.sql.window import Window
# from pyspark.sql.types import *


# @dlt.table(
#     name="drivers",
#     comment="Dados de pilotos limpos e enriquecidos"
# )
# @dlt.expect_all({
#     "valid_driver_id": "driver_id IS NOT NULL",
#     "valid_name": "full_name IS NOT NULL"
# })
# def silver_drivers():
#     """Transforma dados de pilotos da camada bronze"""
#     return (
#         spark.read.table("f1_lakehouse.bronze.drivers_raw")
#         .select(
#             col("driverId").alias("driver_id"),
#             col("number"),
#             col("code"),
#             concat(col("forename"), lit(" "), col("surname")).alias("full_name"),
#             col("forename"),
#             col("surname"),
#             col("nationality"),
#             to_date(col("dateOfBirth"), "yyyy-MM-dd").alias("date_of_birth"),
#             col("_ingested_at"),
#             col("_source"),
#             col("_season")
#         )
#         .filter(col("driver_id").isNotNull())
#         .distinct()
#     )


# @dlt.table(
#     name="constructors",
#     comment="Dados de construtores limpos"
# )
# @dlt.expect_all({
#     "valid_constructor_id": "constructor_id IS NOT NULL",
#     "valid_name": "name IS NOT NULL"
# })
# def silver_constructors():
#     """Transforma dados de construtores da camada bronze"""
#     return (
#         spark.read.table("f1_lakehouse.bronze.constructors_raw")
#         .select(
#             col("constructorId").alias("constructor_id"),
#             col("name"),
#             col("nationality"),
#             col("_ingested_at"),
#             col("_source"),
#             col("_season")
#         )
#         .filter(col("constructor_id").isNotNull())
#         .distinct()
#     )


# @dlt.table(
#     name="races",
#     comment="Dados de corridas limpos"
# )
# @dlt.expect_all({
#     "valid_race_id": "race_id IS NOT NULL",
#     "valid_date": "race_date IS NOT NULL"
# })
# def silver_races():
#     """Transforma dados de corridas da camada bronze"""
#     return (
#         spark.read.table("f1_lakehouse.bronze.races_raw")
#         .select(
#             col("raceId").alias("race_id"),
#             col("year").alias("season"),
#             col("round"),
#             col("circuitId").alias("circuit_id"),
#             col("name").alias("race_name"),
#             to_date(col("date"), "yyyy-MM-dd").alias("race_date"),
#             col("_ingested_at"),
#             col("_source"),
#             col("_season")
#         )
#         .filter(col("race_id").isNotNull())
#         .distinct()
#     )


# @dlt.table(
#     name="results",
#     comment="Dados de resultados limpos"
# )
# @dlt.expect_all({
#     "valid_result_id": "result_id IS NOT NULL",
#     "valid_driver_id": "driver_id IS NOT NULL"
# })
# def silver_results():
#     """Transforma dados de resultados da camada bronze"""
#     return (
#         spark.read.table("f1_lakehouse.bronze.results_raw")
#         .select(
#             col("resultId").alias("result_id"),
#             col("raceId").alias("race_id"),
#             col("driverId").alias("driver_id"),
#             col("constructorId").alias("constructor_id"),
#             col("number").alias("car_number"),
#             col("grid").alias("grid_position"),
#             col("position").alias("final_position"),
#             col("positionText").alias("position_text"),
#             col("points"),
#             col("laps"),
#             col("time").alias("finish_time"),
#             col("milliseconds"),
#             col("fastestLap").alias("fastest_lap_lap"),
#             col("rank").alias("fastest_lap_rank"),
#             col("fastestLapTime").alias("fastest_lap_time"),
#             col("fastestLapSpeed").alias("fastest_lap_speed"),
#             col("statusId").alias("status_id"),
#             col("_season"),
#             col("_ingested_at"),
#             col("_source")
#         )
#         .filter(col("result_id").isNotNull())
#         .filter(col("driver_id").isNotNull())
#         .distinct()
#     )

In [0]:
# @dlt.table(
#     name="silver.races",
#     comment="Dados de corridas limpos e enriquecidos",
#     partition_cols=["season"]
# )
# def silver_races():
#     """
#     Transforma dados de corridas da camada bronze
#     """
#     return (
#         dlt.read("races_raw")
#         .select(
#             col("raceId").alias("race_id"),
#             col("year").alias("season"),
#             col("round"),
#             col("circuitId").alias("circuit_id"),
#             col("name").alias("race_name"),
#             to_date(col("date"), "yyyy-MM-dd").alias("race_date"),
#             col("time").alias("race_time"),
#             to_timestamp(concat(col("date"), lit(" "), col("time")), "yyyy-MM-dd HH:mm:ss").alias("race_timestamp"),
#             col("url"),
#             col("_ingested_at")
#         )
#         .filter(col("race_id").isNotNull())
#     )

In [0]:
# @dlt.table(
#     name="silver.results",
#     comment="Resultados enriquecidos com informações de pilotos e corridas",
#     partition_cols=["season"]
# )
# @dlt.expect_all({
#     "valid_points": "points >= 0",
#     "valid_position": "final_position IS NULL OR final_position > 0"
# })
# def silver_results():
#     """
#     Transforma resultados com joins para enriquecimento
#     """
#     return (
#         dlt.read("results_raw")
#         .join(
#             dlt.read("silver.drivers").alias("drv"),
#             col("driverId") == col("drv.driver_id"),
#             "left"
#         )
#         .join(
#             dlt.read("silver.races").alias("rc"),
#             col("raceId") == col("rc.race_id"),
#             "left"
#         )
#         .select(
#             col("resultId").alias("result_id"),
#             col("raceId").alias("race_id"),
#             col("driverId").alias("driver_id"),
#             col("constructorId").alias("constructor_id"),
#             col("number"),
#             col("grid"),
#             col("position").cast("int").alias("final_position"),
#             col("positionText").alias("position_text"),
#             col("positionOrder").alias("position_order"),
#             col("points").cast("float").alias("points"),
#             col("laps"),
#             col("time").alias("race_time"),
#             col("milliseconds"),
#             col("fastestLap").alias("fastest_lap"),
#             col("rank").alias("fastest_lap_rank"),
#             col("fastestLapTime").alias("fastest_lap_time"),
#             col("fastestLapSpeed").alias("fastest_lap_speed"),
#             col("statusId").alias("status_id"),
#             col("season"),
#             col("rc.race_name"),
#             col("rc.race_date"),
#             col("drv.full_name").alias("driver_name"),
#             col("_ingested_at")
#         )
#         .filter(col("result_id").isNotNull())
#     )